# **Reservoir Capacity Forecasting — Modelling**

## **Objective**
Train and evaluate four forecasting models on the Madrid reservoir system.
Select the best model for production deployment in the Streamlit app.

**Train:** Jan 1998 – Dec 2016 (228 months)  
**Test:** Jan 2017 – Mar 2021 (51 months)

The test period includes the 2017–2018 drought — a meaningful stress test 
for any model.

In [1]:
import json
import pickle
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import Input

from prophet import Prophet
from statsmodels.tsa.arima.model import ARIMA
from xgboost import XGBRegressor
import pmdarima as pm

from src.evaluate import compute_metrics
from src.features import build_xgb_features
from src.forecast import iterative_forecast

I0000 00:00:1780771909.456964   96972 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780771909.458973   96972 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780771909.568497   96972 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780771911.813379   96972 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_PROC    = PROJECT_ROOT / "data" / "processed"
MODELS_DIR   = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

df_pivot = pd.read_csv(DATA_PROC / "reservoirs_pivot.csv", parse_dates=["ds"])
df_pivot = df_pivot.set_index("ds").asfreq("MS")

with open(DATA_PROC / "eda_summary.json") as f:
    eda = json.load(f)

DROUGHT_SEVERE   = eda["drought_thresholds"]["severe_hm3"]
DROUGHT_MODERATE = eda["drought_thresholds"]["moderate_hm3"]

print(f"Shape          : {df_pivot.shape}")
print(f"Date range     : {df_pivot.index.min().date()} → {df_pivot.index.max().date()}")
print(f"Drought severe : {DROUGHT_SEVERE} hm³")
print(f"Drought moderate: {DROUGHT_MODERATE} hm³")

Shape          : (279, 16)
Date range     : 1998-01-01 → 2021-03-01
Drought severe : 488.5 hm³
Drought moderate: 573.6 hm³


## **1. Train / Test Split**

Chronological split — never shuffle time series data. Shuffling would leak 
future information into training and produce optimistically biased metrics.

80/20 split at December 2016.

In [3]:
# 80/20 split — keep last 20% for evaluation
# Never shuffle time series — always chronological
TRAIN_END = "2016-12-01"
TEST_START = "2017-01-01"

train = df_pivot.loc[:TRAIN_END, "total_hm3"]
test  = df_pivot.loc[TEST_START:, "total_hm3"]

print(f"Train : {train.index.min().date()} → {train.index.max().date()}  ({len(train)} months)")
print(f"Test  : {test.index.min().date()}  → {test.index.max().date()}   ({len(test)} months)")

# Visualise split
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=train.index, y=train.values,
    mode="lines", name="Train",
    line=dict(color="#4a9eff", width=2),
))
fig.add_trace(go.Scatter(
    x=test.index, y=test.values,
    mode="lines", name="Test",
    line=dict(color="#ff9944", width=2),
))
fig.add_vline(
    x=pd.Timestamp(TEST_START).timestamp() * 1000,
    line=dict(color="#666", dash="dash", width=1),
    annotation_text="Train / Test split",
    annotation_font=dict(color="#666"),
)
fig.update_layout(
    title="Train / Test Split",
    xaxis_title="Date",
    yaxis_title="Capacity (hm³)",
    template="plotly_dark",
    height=400,
    hovermode="x unified",
    width=1200,
)
fig.show()

Train : 1998-01-01 → 2016-12-01  (228 months)
Test  : 2017-01-01  → 2021-03-01   (51 months)


## **2. Metrics**

All models evaluated on the same test set with four metrics:
- **R²** — proportion of variance explained
- **MAE** — mean absolute error in hm³
- **RMSE** — root mean squared error, penalises large errors more
- **MAPE** — mean absolute percentage error, scale-independent

MAPE is the primary selection criterion — it is interpretable as accuracy 
and comparable across datasets.

In [4]:
results = {}
print("Model          R²        MAE        RMSE       MAPE      Accuracy")
print("-" * 70)

Model          R²        MAE        RMSE       MAPE      Accuracy
----------------------------------------------------------------------


## **3. Prophet**

Facebook Prophet decomposes the series into trend + seasonality + holidays.
Configured with yearly seasonality only — no weekly or daily components for 
monthly data.

**Result: MAPE 17.2%, R²=-0.43**

Prophet captures the seasonal shape but overestimates amplitude in the test 
period. The model assumes the future will look like the past — it struggles 
with the lower-amplitude 2017–2021 period after a wetter 2013–2016.

In [5]:
# Prepare Prophet format — requires ds and y columns
df_prophet = df_pivot["total_hm3"].reset_index()
df_prophet.columns = ["ds", "y"]
df_prophet = df_prophet.dropna()

prophet_train = df_prophet[df_prophet["ds"] <= TRAIN_END]
prophet_test  = df_prophet[df_prophet["ds"] >= TEST_START]

# Fit
model_prophet = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode="additive",
)
model_prophet.fit(prophet_train)

# Predict on test period
future   = model_prophet.make_future_dataframe(periods=len(prophet_test), freq="MS")
forecast = model_prophet.predict(future)

# Evaluate on test only
pred_prophet = forecast[forecast["ds"] >= TEST_START]["yhat"].values
true_prophet = prophet_test["y"].values

metrics_prophet = compute_metrics(true_prophet, pred_prophet, "Prophet")
results["Prophet"] = {
    **metrics_prophet,
    "_forecast": forecast,
    "_train":    prophet_train,
    "_test":     prophet_test,
}

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=prophet_train["ds"], y=prophet_train["y"],
    mode="lines", name="Train",
    line=dict(color="#4a9eff", width=1.5),
))
fig.add_trace(go.Scatter(
    x=prophet_test["ds"], y=prophet_test["y"],
    mode="lines", name="Test (actual)",
    line=dict(color="#ff9944", width=2),
))
fig.add_trace(go.Scatter(
    x=forecast[forecast["ds"] >= TEST_START]["ds"],
    y=pred_prophet,
    mode="lines", name="Prophet forecast",
    line=dict(color="#44ff88", width=2, dash="dash"),
))
fig.add_trace(go.Scatter(
    x=list(forecast[forecast["ds"] >= TEST_START]["ds"]) +
      list(forecast[forecast["ds"] >= TEST_START]["ds"][::-1]),
    y=list(forecast[forecast["ds"] >= TEST_START]["yhat_upper"]) +
      list(forecast[forecast["ds"] >= TEST_START]["yhat_lower"][::-1]),
    fill="toself",
    fillcolor="rgba(68,255,136,0.1)",
    line=dict(color="rgba(0,0,0,0)"),
    name="95% CI",
))
fig.add_vline(
    x=pd.Timestamp(TEST_START).timestamp() * 1000,
    line=dict(color="#666", dash="dash", width=1),
)
fig.update_layout(
    title="Prophet — Forecast vs Actual",
    xaxis_title="Date", yaxis_title="Capacity (hm³)",
    template="plotly_dark", height=450, hovermode="x unified",
    width=1200,
)
fig.show()

20:51:54 - cmdstanpy - INFO - Chain [1] start processing
20:51:54 - cmdstanpy - INFO - Chain [1] done processing


Prophet        R²=-0.4256  MAE=  103.3  RMSE=  127.0  MAPE=17.21%  Accuracy=82.79%


In [6]:
with open(MODELS_DIR / "prophet_bundle.pkl", "wb") as f:
    pickle.dump({
        "model":      model_prophet,
        "metrics":    {k: v for k, v in metrics_prophet.items() if not k.startswith("_")},
        "train_end":  TRAIN_END,
        "test_start": TEST_START,
    }, f)

print(f"prophet_bundle.pkl — {(MODELS_DIR / 'prophet_bundle.pkl').stat().st_size / 1024:.0f} KB")

prophet_bundle.pkl — 29 KB


## **4. SARIMAX**

Seasonal ARIMA with order (2,0,1)(1,1,1,12) derived from the ACF/PACF analysis 
in the EDA notebook. Seasonal differencing (D=1) removes the annual cycle.

**Result: MAPE 11.9%, R²=0.30**

Better than Prophet — captures the seasonal structure correctly. The confidence 
interval widens significantly beyond 12 months, reflecting the model's uncertainty 
over longer horizons.

In [7]:
# SARIMAX order from EDA ACF/PACF analysis
ORDER          = tuple(eda["sarimax_order"]["order"])
SEASONAL_ORDER = tuple(eda["sarimax_order"]["seasonal_order"])

model_sarimax = ARIMA(
    train,
    order=ORDER,
    seasonal_order=SEASONAL_ORDER,
)
fit_sarimax = model_sarimax.fit()

# Forecast test period
forecast_sarimax = fit_sarimax.get_forecast(steps=len(test))
pred_sarimax     = forecast_sarimax.predicted_mean.values
ci_sarimax       = forecast_sarimax.conf_int()
true_sarimax     = test.values

metrics_sarimax = compute_metrics(true_sarimax, pred_sarimax, "SARIMAX")
results["SARIMAX"] = {
    **metrics_sarimax,
    "_pred":  pred_sarimax,
    "_ci":    ci_sarimax,
    "_dates": test.index,
}

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=train.index, y=train.values,
    mode="lines", name="Train",
    line=dict(color="#4a9eff", width=1.5),
))
fig.add_trace(go.Scatter(
    x=test.index, y=test.values,
    mode="lines", name="Test (actual)",
    line=dict(color="#ff9944", width=2),
))
fig.add_trace(go.Scatter(
    x=test.index, y=pred_sarimax,
    mode="lines", name="SARIMAX forecast",
    line=dict(color="#44ff88", width=2, dash="dash"),
))
fig.add_trace(go.Scatter(
    x=list(test.index) + list(test.index[::-1]),
    y=list(ci_sarimax.iloc[:, 0]) + list(ci_sarimax.iloc[:, 1][::-1]),
    fill="toself",
    fillcolor="rgba(68,255,136,0.1)",
    line=dict(color="rgba(0,0,0,0)"),
    name="95% CI",
))
fig.add_vline(
    x=pd.Timestamp(TEST_START).timestamp() * 1000,
    line=dict(color="#666", dash="dash", width=1),
)
fig.update_layout(
    title=f"SARIMAX{ORDER}{SEASONAL_ORDER} — Forecast vs Actual",
    xaxis_title="Date", yaxis_title="Capacity (hm³)",
    template="plotly_dark", height=450, hovermode="x unified",
    width=1200,
)
fig.show()

SARIMAX        R²=0.2981  MAE=   74.7  RMSE=   89.1  MAPE=11.91%  Accuracy=88.09%


## **5. Bidirectional LSTM**

Deep learning approach — reads the sequence both forwards and backwards.
Architecture: BiLSTM(64) → Dropout(0.3) → BiLSTM(32) → Dropout(0.3) → Dense(1)

Features: scaled capacity + cyclic month encoding (sin/cos) + 3 lag features.
Window size: 24 months. Train/val/test: 70/15/15.
Early stopping with patience=25 — stopped at epoch 77.

**Result: MAPE 6.7%, R²=0.73**

Significant improvement over statistical models. The bidirectional architecture 
captures both the approach to seasonal peaks and the descent from them. 
Dropout regularisation prevents overfitting on the small dataset.

In [8]:
# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

WINDOW_SIZE = 24

# Prepare features
df_lstm = df_pivot[["total_hm3"]].copy()
df_lstm["month_num"] = df_pivot.index.month
df_lstm["month_sin"] = np.sin(2 * np.pi * df_lstm["month_num"] / 12)
df_lstm["month_cos"] = np.cos(2 * np.pi * df_lstm["month_num"] / 12)

# Scale target
scaler_target = MinMaxScaler()
df_lstm["total_scaled"] = scaler_target.fit_transform(df_lstm[["total_hm3"]])

# Lag features
for lag in range(1, 4):
    df_lstm[f"lag_{lag}"] = df_lstm["total_scaled"].shift(lag)

df_lstm = df_lstm.dropna()

# Scale features
FEATURE_COLS = ["total_scaled", "month_sin", "month_cos", "lag_1", "lag_2", "lag_3"]
scaler_feats = MinMaxScaler()
df_lstm[FEATURE_COLS] = scaler_feats.fit_transform(df_lstm[FEATURE_COLS])

# Create windowed dataset
def create_dataset(features, targets, window_size):
    X, y = [], []
    for i in range(len(targets) - window_size):
        X.append(features[i:i + window_size])
        y.append(targets[i + window_size])
    return np.array(X), np.array(y)

features = df_lstm[FEATURE_COLS].values
targets  = df_lstm["total_scaled"].values

X_all, y_all = create_dataset(features, targets, WINDOW_SIZE)
dates_all    = df_lstm.index[WINDOW_SIZE:]

# Split aligned with train/test dates
train_mask = dates_all <= TRAIN_END
test_mask  = dates_all >= TEST_START

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_test,  y_test  = X_all[test_mask],  y_all[test_mask]
test_dates        = dates_all[test_mask]

# Validation from last 15% of train
val_split = int(len(X_train) * 0.85)
X_tr, X_val = X_train[:val_split], X_train[val_split:]
y_tr, y_val = y_train[:val_split], y_train[val_split:]

print(f"Train  : {len(X_tr)} samples")
print(f"Val    : {len(X_val)} samples")
print(f"Test   : {len(X_test)} samples")

Train  : 170 samples
Val    : 31 samples
Test   : 51 samples


In [9]:
model_lstm = Sequential([
    Input(shape=(WINDOW_SIZE, len(FEATURE_COLS))),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(32)),
    Dropout(0.3),
    Dense(1),
])

model_lstm.compile(optimizer="adam", loss="mse")

callbacks = [
    EarlyStopping(monitor="val_loss", patience=25, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=10, verbose=0),
]

history = model_lstm.fit(
    X_tr, y_tr,
    epochs=150,
    batch_size=16,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=0,
)

print(f"Stopped at epoch : {len(history.history['loss'])}")
print(f"Best val_loss    : {min(history.history['val_loss']):.6f}")

# Training curve
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=history.history["loss"],
    mode="lines", name="Train loss",
    line=dict(color="#4a9eff"),
))
fig.add_trace(go.Scatter(
    y=history.history["val_loss"],
    mode="lines", name="Val loss",
    line=dict(color="#ff9944"),
))
fig.update_layout(
    title="LSTM — Training Loss",
    xaxis_title="Epoch", yaxis_title="MSE Loss",
    template="plotly_dark", height=350,
    width=1200,
)
fig.show()

E0000 00:00:1780771916.586279   96972 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1780771916.587037   97070 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1780771916.604762   96972 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Stopped at epoch : 101
Best val_loss    : 0.002840


In [10]:
# Predict on test set
y_pred_scaled = model_lstm(X_test, training=False).numpy()
y_pred_lstm   = scaler_target.inverse_transform(y_pred_scaled).flatten()
y_true_lstm   = scaler_target.inverse_transform(y_test.reshape(-1, 1)).flatten()

metrics_lstm = compute_metrics(y_true_lstm, y_pred_lstm, "LSTM")
results["LSTM"] = {
    **metrics_lstm,
    "_pred":  y_pred_lstm,
    "_true":  y_true_lstm,
    "_dates": test_dates,
}

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=train.index, y=train.values,
    mode="lines", name="Train",
    line=dict(color="#4a9eff", width=1.5),
))
fig.add_trace(go.Scatter(
    x=test_dates, y=y_true_lstm,
    mode="lines", name="Test (actual)",
    line=dict(color="#ff9944", width=2),
))
fig.add_trace(go.Scatter(
    x=test_dates, y=y_pred_lstm,
    mode="lines", name="LSTM forecast",
    line=dict(color="#44ff88", width=2, dash="dash"),
))
fig.add_vline(
    x=pd.Timestamp(TEST_START).timestamp() * 1000,
    line=dict(color="#666", dash="dash", width=1),
)
fig.update_layout(
    title="Bidirectional LSTM — Forecast vs Actual",
    xaxis_title="Date", yaxis_title="Capacity (hm³)",
    template="plotly_dark", height=450, hovermode="x unified",
    width=1200,
)
fig.show()

LSTM           R²=0.7492  MAE=   41.2  RMSE=   53.2  MAPE=6.11%  Accuracy=93.89%


## **6. XGBoost**

Gradient boosted trees with engineered temporal features:
- 12 lag features (lag_1 to lag_12)
- Rolling statistics (mean and std over 3, 6, 12 months)
- Cyclic month encoding (sin/cos) + raw month number

**Result: MAPE 4.6%, R²=0.82**

Best performing model. Feature importance reveals that `rolling_mean_3` (0.41) 
and `lag_1` (0.29) account for 70% of predictive power — the system has strong 
short-term momentum. XGBoost outperforms LSTM because with only 279 data points, 
well-engineered features beat deep learning.

In [11]:
df_xgb = build_xgb_features(df_pivot["total_hm3"])

FEATURE_COLS_XGB = [c for c in df_xgb.columns if c != "y"]

X_xgb = df_xgb[FEATURE_COLS_XGB]
y_xgb = df_xgb["y"]

train_mask_xgb = df_xgb.index <= TRAIN_END
test_mask_xgb  = df_xgb.index >= TEST_START

X_train_xgb = X_xgb[train_mask_xgb]
y_train_xgb = y_xgb[train_mask_xgb]
X_test_xgb  = X_xgb[test_mask_xgb]
y_test_xgb  = y_xgb[test_mask_xgb]

print(f"Train: {len(X_train_xgb)} samples")
print(f"Test : {len(X_test_xgb)} samples")
print(f"Features: {len(FEATURE_COLS_XGB)}")

Train: 216 samples
Test : 51 samples
Features: 20


In [12]:
model_xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0,
)

model_xgb.fit(
    X_train_xgb, y_train_xgb,
    eval_set=[(X_test_xgb, y_test_xgb)],
    verbose=False,
)

pred_xgb = model_xgb.predict(X_test_xgb)
metrics_xgb = compute_metrics(y_test_xgb.values, pred_xgb, "XGBoost")
results["XGBoost"] = {
    **metrics_xgb,
    "_pred":  pred_xgb,
    "_dates": X_test_xgb.index,
}

# Feature importance
importance = pd.Series(
    model_xgb.feature_importances_,
    index=FEATURE_COLS_XGB
).sort_values(ascending=False)

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "XGBoost — Forecast vs Actual",
    "Feature Importance",
))

# Forecast
fig.add_trace(go.Scatter(
    x=test.index, y=test.values,
    mode="lines", name="Actual",
    line=dict(color="#ff9944", width=2),
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=X_test_xgb.index, y=pred_xgb,
    mode="lines", name="XGBoost",
    line=dict(color="#44ff88", width=2, dash="dash"),
), row=1, col=1)

# Importance
fig.add_trace(go.Bar(
    x=importance.values[:10],
    y=importance.index[:10],
    orientation="h",
    marker_color="#4a9eff",
    showlegend=False,
), row=1, col=2)

fig.update_layout(
    template="plotly_dark", height=400,
    title="XGBoost — Forecast & Feature Importance",
    width=1800,
)
fig.show()

XGBoost        R²=0.8177  MAE=   31.1  RMSE=   45.4  MAPE=4.64%  Accuracy=95.36%


## **7. Model Comparison**

| Model       | R²    | MAE   | RMSE  | MAPE  | Accuracy |
|-------------|-------|-------|-------|-------|----------|
| XGBoost     | 0.82  | 31.1  | 45.4  | 4.6%  | 95.4%    |
| LSTM        | 0.73  | 44.4  | 55.1  | 6.7%  | 93.3%    |
| SARIMAX     | 0.30  | 74.7  | 89.1  | 11.9% | 88.1%    |
| Prophet     | -0.43 | 103.4 | 127.0 | 17.2% | 82.8%    |

**XGBoost selected for production** — best metrics, 726 KB serialised, 
fast inference, interpretable feature importance.

In [13]:
# Model comparison table
comparison = pd.DataFrame([
    {k: v for k, v in results[m].items() if not k.startswith("_")}
    for m in ["Prophet", "SARIMAX", "LSTM", "XGBoost"]
])

print("\nModel Comparison — Test Set (2017–2021)")
print("=" * 70)
print(comparison.to_string(index=False))

# Visual comparison
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=test.index, y=test.values,
    mode="lines", name="Actual",
    line=dict(color="#ffffff", width=2),
))
fig.add_trace(go.Scatter(
    x=results["Prophet"]["_test"]["ds"],
    y=results["Prophet"]["_forecast"][
        results["Prophet"]["_forecast"]["ds"] >= TEST_START
    ]["yhat"].values,
    mode="lines", name=f"Prophet (MAPE {results['Prophet']['mape']:.1f}%)",
    line=dict(color="#ff4444", width=1.5, dash="dash"),
))
fig.add_trace(go.Scatter(
    x=results["SARIMAX"]["_dates"],
    y=results["SARIMAX"]["_pred"],
    mode="lines", name=f"SARIMAX (MAPE {results['SARIMAX']['mape']:.1f}%)",
    line=dict(color="#ffaa44", width=1.5, dash="dash"),
))
fig.add_trace(go.Scatter(
    x=results["LSTM"]["_dates"],
    y=results["LSTM"]["_pred"],
    mode="lines", name=f"LSTM (MAPE {results['LSTM']['mape']:.1f}%)",
    line=dict(color="#44ff88", width=2, dash="dash"),
))
fig.add_trace(go.Scatter(
    x=results["XGBoost"]["_dates"],
    y=results["XGBoost"]["_pred"],
    mode="lines", name=f"XGBoost (MAPE {results['XGBoost']['mape']:.1f}%)",
    line=dict(color="#4488ff", width=2, dash="dash"),
))

fig.update_layout(
    title="Model Comparison — Test Period 2017–2021",
    xaxis_title="Date", yaxis_title="Capacity (hm³)",
    template="plotly_dark", height=450, hovermode="x unified",
    width=1200,
)
fig.show()


Model Comparison — Test Set (2017–2021)
  model      r2    mae   rmse  mape  accuracy
Prophet -0.4256 103.35 126.98 17.21     82.79
SARIMAX  0.2981  74.72  89.10 11.91     88.09
   LSTM  0.7492  41.24  53.25  6.11     93.89
XGBoost  0.8177  31.12  45.41  4.64     95.36


In [14]:
# LSTM — save weights + scalers + config
lstm_bundle = {
    "model_weights":   model_lstm.get_weights(),
    "model_config":    model_lstm.get_config(),
    "scaler_target":   scaler_target,
    "scaler_feats":    scaler_feats,
    "feature_cols":    FEATURE_COLS,
    "window_size":     WINDOW_SIZE,
    "train_end":       TRAIN_END,
    "metrics": {
        "r2":       metrics_lstm["r2"],
        "mae":      metrics_lstm["mae"],
        "rmse":     metrics_lstm["rmse"],
        "mape":     metrics_lstm["mape"],
        "accuracy": metrics_lstm["accuracy"],
    },
    "drought_thresholds": {
        "severe_hm3":   DROUGHT_SEVERE,
        "moderate_hm3": DROUGHT_MODERATE,
    },
    "df_lstm_index": df_lstm.index.tolist(),
}

with open(MODELS_DIR / "lstm_bundle.pkl", "wb") as f:
    pickle.dump(lstm_bundle, f)

# SARIMAX
with open(MODELS_DIR / "sarimax_bundle.pkl", "wb") as f:
    pickle.dump({
        "model":   fit_sarimax,
        "order":   ORDER,
        "seasonal_order": SEASONAL_ORDER,
        "metrics": {k: v for k, v in metrics_sarimax.items() if not k.startswith("_")},
    }, f)

xgb_bundle = {
    "model":          model_xgb,
    "feature_cols":   FEATURE_COLS_XGB,
    "n_lags":         12,
    "metrics": {k: v for k, v in metrics_xgb.items() if not k.startswith("_")},
    "drought_thresholds": {
        "severe_hm3":   DROUGHT_SEVERE,
        "moderate_hm3": DROUGHT_MODERATE,
    },
    "feature_importance": importance.to_dict(),
}

with open(MODELS_DIR / "xgb_bundle.pkl", "wb") as f:
    pickle.dump(xgb_bundle, f)

# Model comparison
comparison.to_csv(MODELS_DIR / "model_comparison.csv", index=False)

print("Exported:")
print(f"  lstm_bundle.pkl    — {(MODELS_DIR / 'lstm_bundle.pkl').stat().st_size / 1024:.0f} KB")
print(f"  sarimax_bundle.pkl — {(MODELS_DIR / 'sarimax_bundle.pkl').stat().st_size / 1024:.0f} KB")
print(f"  xgb_bundle.pkl     — {(MODELS_DIR / 'xgb_bundle.pkl').stat().st_size / 1024:.0f} KB")
print(f"  model_comparison.csv — {(MODELS_DIR / 'model_comparison.csv').stat().st_size / 1024:.0f} KB")

Exported:
  lstm_bundle.pkl    — 315 KB
  sarimax_bundle.pkl — 23340 KB
  xgb_bundle.pkl     — 726 KB
  model_comparison.csv — 0 KB


## **8. Short-Term Forecast — 24 Months**

Iterative forecast Apr 2021 → Mar 2023.

Each predicted month is appended to the series and used to generate features 
for the next prediction. Short horizon limits error accumulation.

**Results:**
- Mean predicted: 699.7 hm³
- Min predicted: 620.0 hm³ (Mar 2022)
- Max predicted: 822.8 hm³ (Apr 2021)
- Drought risk: 0 months severe, 0 months moderate

The forecast follows a realistic seasonal pattern — peaks in spring, 
troughs in autumn — consistent with the historical record.

In [ ]:
# Iterative forecast — 24 months from April 2021
N_MONTHS = 24

# Start from last known data
series_extended = df_pivot["total_hm3"].copy()
last_date       = series_extended.index[-1]

future_preds = []
future_dates = []

future_dates, future_preds = iterative_forecast(
    series       = df_pivot["total_hm3"],
    model        = model_xgb,
    feature_cols = FEATURE_COLS_XGB,
    n_lags       = 12,
    n_months     = N_MONTHS,
)

future_dates = pd.DatetimeIndex(future_dates)

# Plot — last 3 years of history + forecast
hist_start = "2018-01-01"
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_pivot.loc[hist_start:].index,
    y=df_pivot.loc[hist_start:, "total_hm3"].values,
    mode="lines", name="Historical",
    line=dict(color="#4a9eff", width=2),
))
fig.add_trace(go.Scatter(
    x=future_dates, y=future_preds,
    mode="lines+markers", name="XGBoost forecast",
    line=dict(color="#44ff88", width=2, dash="dash"),
    marker=dict(size=5),
))

# Drought thresholds
fig.add_hline(y=DROUGHT_SEVERE,   line=dict(color="#ff4444", dash="dot", width=1),
                annotation_text=f"Severe drought {DROUGHT_SEVERE:.0f} hm³",
                annotation_font=dict(color="#ff4444", size=10))
fig.add_hline(y=DROUGHT_MODERATE, line=dict(color="#ffaa44", dash="dot", width=1),
                annotation_text=f"Moderate drought {DROUGHT_MODERATE:.0f} hm³",
                annotation_font=dict(color="#ffaa44", size=10))

fig.add_vline(
    x=last_date.timestamp() * 1000,
    line=dict(color="#666", dash="dash", width=1),
    annotation_text="Forecast start",
    annotation_font=dict(color="#666"),
)

fig.update_layout(
    title=f"XGBoost — 24-Month Forecast from {last_date.strftime('%b %Y')}",
    xaxis_title="Date", yaxis_title="Capacity (hm³)",
    template="plotly_dark", height=450, hovermode="x unified",
    width=1200,
)
fig.show()

print(f"\nForecast {future_dates[0].strftime('%b %Y')} → {future_dates[-1].strftime('%b %Y')}")
print(f"Mean predicted : {np.mean(future_preds):,.1f} hm³")
print(f"Min predicted  : {np.min(future_preds):,.1f} hm³  ({future_dates[np.argmin(future_preds)].strftime('%b %Y')})")
print(f"Max predicted  : {np.max(future_preds):,.1f} hm³  ({future_dates[np.argmax(future_preds)].strftime('%b %Y')})")

# Drought risk assessment
n_severe   = sum(1 for p in future_preds if p < DROUGHT_SEVERE)
n_moderate = sum(1 for p in future_preds if DROUGHT_SEVERE <= p < DROUGHT_MODERATE)
print(f"\nDrought risk (next 24 months):")
print(f"  Severe   : {n_severe} months")
print(f"  Moderate : {n_moderate} months")
print(f"  Normal   : {N_MONTHS - n_severe - n_moderate} months")


Forecast Apr 2021 → Mar 2023
Mean predicted : 699.7 hm³
Min predicted  : 620.0 hm³  (Mar 2022)
Max predicted  : 822.8 hm³  (Apr 2021)

Drought risk (next 24 months):
  Severe   : 0 months
  Moderate : 0 months
  Normal   : 24 months


## **9. Long-Term Forecast — 12 Years**

Iterative forecast Apr 2021 → Mar 2033 (144 months).

**Results:**
- Mean predicted: 682.9 hm³
- Min predicted: 563.9 hm³ (Dec 2025)
- Max predicted: 822.8 hm³ (Apr 2021)
- Drought risk: 2 months moderate, 0 months severe

The series maintains stable seasonality over 144 iterations without diverging —
evidence that the rolling_mean features act as a natural stabiliser, keeping 
predictions anchored to the historical distribution.

Caveat: iterative forecasts accumulate error at each step. Beyond 12–24 months 
the prediction should be interpreted as a plausible scenario based on historical 
patterns, not a precise estimate.

In [16]:
# Iterative forecast — 144 months from April 2021
N_MONTHS = 144

future_dates, future_preds = iterative_forecast(
    series       = df_pivot["total_hm3"],
    model        = model_xgb,
    feature_cols = FEATURE_COLS_XGB,
    n_lags       = 12,
    n_months     = N_MONTHS,
)

# Plot — last 3 years of history + forecast
hist_start = "2018-01-01"
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_pivot.loc[hist_start:].index,
    y=df_pivot.loc[hist_start:, "total_hm3"].values,
    mode="lines", name="Historical",
    line=dict(color="#4a9eff", width=2),
))
fig.add_trace(go.Scatter(
    x=future_dates, y=future_preds,
    mode="lines+markers", name="XGBoost forecast",
    line=dict(color="#44ff88", width=2, dash="dash"),
    marker=dict(size=5),
))

fig.add_hline(y=DROUGHT_SEVERE,   line=dict(color="#ff4444", dash="dot", width=1),
                annotation_text=f"Severe drought {DROUGHT_SEVERE:.0f} hm³",
                annotation_font=dict(color="#ff4444", size=10))
fig.add_hline(y=DROUGHT_MODERATE, line=dict(color="#ffaa44", dash="dot", width=1),
                annotation_text=f"Moderate drought {DROUGHT_MODERATE:.0f} hm³",
                annotation_font=dict(color="#ffaa44", size=10))

fig.add_vline(
    x=df_pivot.index[-1].timestamp() * 1000,
    line=dict(color="#666", dash="dash", width=1),
    annotation_text="Forecast start",
    annotation_font=dict(color="#666"),
)

fig.update_layout(
    title=f"XGBoost — 12-Year Forecast from {df_pivot.index[-1].strftime('%b %Y')}",
    xaxis_title="Date", yaxis_title="Capacity (hm³)",
    template="plotly_dark", height=450, hovermode="x unified",
)
fig.show()

print(f"\nForecast {future_dates[0].strftime('%b %Y')} → {future_dates[-1].strftime('%b %Y')}")
print(f"Mean predicted : {np.mean(future_preds):,.1f} hm³")
print(f"Min predicted  : {np.min(future_preds):,.1f} hm³  ({future_dates[np.argmin(future_preds)].strftime('%b %Y')})")
print(f"Max predicted  : {np.max(future_preds):,.1f} hm³  ({future_dates[np.argmax(future_preds)].strftime('%b %Y')})")

n_severe   = sum(1 for p in future_preds if p < DROUGHT_SEVERE)
n_moderate = sum(1 for p in future_preds if DROUGHT_SEVERE <= p < DROUGHT_MODERATE)
print(f"Drought risk (next {N_MONTHS} months / {N_MONTHS//12} years):")
print(f"  Severe   : {n_severe} months")
print(f"  Moderate : {n_moderate} months")
print(f"  Normal   : {N_MONTHS - n_severe - n_moderate} months")


Forecast Apr 2021 → Mar 2033
Mean predicted : 682.9 hm³
Min predicted  : 563.9 hm³  (Dec 2025)
Max predicted  : 822.8 hm³  (Apr 2021)
Drought risk (next 144 months / 12 years):
  Severe   : 0 months
  Moderate : 2 months
  Normal   : 142 months


In [17]:
xgb_bundle["forecast"] = {
    "dates":        [d.strftime("%Y-%m-%d") for d in future_dates],
    "predictions":  [round(p, 2) for p in future_preds],
    "n_months":     N_MONTHS,
}

with open(MODELS_DIR / "xgb_bundle.pkl", "wb") as f:
    pickle.dump(xgb_bundle, f)

print("xgb_bundle.pkl updated with forecast")

xgb_bundle.pkl updated with forecast
